In [112]:
with open("input.txt", "r") as f:
    text = f.read()

In [113]:
length = len(text)
print(f"Length of text: {length} characters")
char = sorted(list(set(text)))
vocab_size = len(char)
print("".join(char))
print(f"Vocab size: {vocab_size}")

Length of text: 1115394 characters

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


In [114]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])
print(encode("hello world"))
print(decode(encode("hello world"))) # note this is one of the simplest possible tokenizers, it just maps each character to an integer. everyone has their own tokenizer like google use sentencepiece, openai use bpe, etc. we will build our own tokenizer in the next notebook.

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [115]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [116]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [117]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(train_data.shape, val_data.shape)

torch.Size([1003854]) torch.Size([111540])


In [118]:
block_size = 8
train_data[:block_size+1] # we will use the first 8 characters to predict
print(train_data[:block_size+1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [119]:
#use mps as i am using the mac with m4 
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
n_embeed = 384 
max_iters = 20000
eval_iters = 2000 
lr_rate = 2e-4
dropout = 0.2
n_layer = 8
n_head = 8
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"using {device} device")

using mps device


In [120]:
torch.manual_seed(1337)
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)
print("\ntargets:")
print(yb.shape)
print(yb)

inputs:
torch.Size([64, 256])
tensor([[ 0, 26, 53,  ..., 56, 43, 47],
        [60, 43, 56,  ..., 56,  1, 41],
        [26, 21, 33,  ..., 26, 21, 13],
        ...,
        [ 5, 57,  1,  ...,  1, 35, 47],
        [56, 53, 53,  ..., 59, 50, 42],
        [42, 47, 56,  ..., 39, 56,  1]], device='mps:0')

targets:
torch.Size([64, 256])
tensor([[26, 53, 58,  ..., 43, 47, 45],
        [43, 56,  1,  ...,  1, 41, 53],
        [21, 33, 31,  ..., 21, 13, 10],
        ...,
        [57,  1, 52,  ..., 35, 47, 50],
        [53, 53, 58,  ..., 50, 42,  1],
        [47, 56, 43,  ..., 56,  1, 51]], device='mps:0')


In [121]:
class Head(torch.nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embeed, head_size, bias=False)
        self.query = nn.Linear(n_embeed, head_size, bias=False)
        self.value = nn.Linear(n_embeed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) # (B,T,16)
        q = self.query(x) # (B,T,16)
        v = self.value(x) # (B,T,16)
        weights = q @ k.transpose(-2, -1) * (self.head_size ** -0.5) # (B,T,16) @ (B,16,T) -> (B,T,T)
        tril = torch.tril(torch.ones(T, T , device=x.device))
        weights = weights.masked_fill(tril == 0, float('-inf')) # when we talk about the encoder transformer we remove this bcz we want to attend to all the tokens in the input sequence, but in the decoder transformer we want to attend only to the previous tokens in the output sequence, so we use this mask to prevent the model from attending to future tokens.
        weights = torch.softmax(weights, dim=-1)
        weights = self.dropout(weights)
        out = weights @ v # (B,T,T) @ (B,T,C) -> (B,T,C)
        return out
    

In [122]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = torch.nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = torch.nn.Linear(n_embeed, n_embeed)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [123]:
class feedforward(torch.nn.Module):
    def __init__(self, n_embeed):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_embeed, 4*n_embeed), #according to paper there is the multiplier of 4 in the hidden layers 
            torch.nn.ReLU(),
            torch.nn.Linear(4*n_embeed, n_embeed),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

In [124]:
class Block(torch.nn.Module):
    def __init__(self, n_embeed , n_head):
        super().__init__()
        head_size = n_embeed // n_head 
        self.sa_head = MultiHeadAttention(num_heads=n_head, head_size=head_size)
        self.ffwd = feedforward(n_embeed)
        self.ln1 = nn.LayerNorm(n_embeed)
        self.ln2 = nn.LayerNorm(n_embeed)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = x+self.sa_head(self.ln1(x))#this is slightly deviation from the original paper as we are passing the layer norem before the multi head attention and feedforward, but it is a common practice to do so, and it works better than the original paper.
        x = x+self.ffwd(self.ln2(x))
        return x

In [125]:
class BigramLanguageModel(torch.nn.Module):
    def __init__(self, vocab_size, n_embeed):
        super().__init__()
        self.token_embedding_table = torch.nn.Embedding(vocab_size, n_embeed) 
        self.position_embedding_table = torch.nn.Embedding(block_size, n_embeed)
        #not giving the good enhaced result when we try multiple Block() , deep neural net suffer from the optimisation issue 
        # self.blocks = nn.Sequential(
        #     Block(n_embeed, n_head=4),
        #     Block(n_embeed, n_head=4),
        #     Block(n_embeed, n_head=4),
        #     nn.LayerNorm(n_embeed),
        # )
        self.blocks = nn.Sequential(*[Block(n_embeed, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embeed)
        self.lm_head = torch.nn.Linear(n_embeed, vocab_size)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        token_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(idx.shape[1], device=idx.device)) # (T,C)
        x = token_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # becomes (B,C) , as we only want to provide the last character as the input to predict the next character
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1) # (B,T+1)
        return idx

In [126]:
model = BigramLanguageModel(vocab_size, n_embeed)
model = model.to(device)
logits, loss = model(xb, yb)
print("logits shape:", logits.shape)
print("loss:", loss.item())
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))

logits shape: torch.Size([16384, 65])
loss: 4.277037620544434

tRNt'OUWzNdaNv;DZ!HWJxsg-rG$l.
VXx;h&CEqoyJOlF.DmdMw;u;cjEIgcOQOID;$wig.tRIgazPSVyRpKBE-3UQBdJ'AIIxX


In [127]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_rate)
for steps in range(max_iters):
    if steps % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



step 0: train loss 4.2785, val loss 4.2821


In [ ]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=1000)[0].tolist()))



DUKE VINCENTIO:
Stand brother, sir, here it, uncle he got.

VIRGILIA:
A dog of the yousician, let your good brother, sister,
nor it to die.

VOLUMNIA:
She is in the mar, and the matter:
there is! What say you, Juliet alone and bird.
Is thy life?

JULIET:
Being a child! prompt fear: speak, and look fellow good?

FLORIZEL:
And rumour, by my man's tooth made.

JULIET:
Ay, if you doth make leave your retires,
A mother tempt my todder dial should have
So dear and let me 'gainst words out again,
Savest with honour's princes to hear throught of,
His perdom of preserve
Is posterity and secut
No god costerb: shall be more the Capitol,
But court did this hoursest, do begg them buried.
His apple and dreams on daughter, and we will,
He were laddy's wounds. O mother!
Dread!
In it the whitest through thee grief: why, general,
My heart play'd many fellows upon him.

FRIAR LAURENCE:
For traitor the mind: what the journey, rise!
I serve, or I know the senate, and let my indeed
Will on brave it so lon

In [ ]:
print(sum(p.numel() for p in model.parameters())/1e6, "M parameters")

10.788929 M parameters


In [ ]:
import torch
torch.save(model.state_dict(), "shakespeare_transformer.pt")